# Fase 4 · Modelos finales y fusión

**Detección de exoplanetas en Kepler con una red multirrama — iteración 2** (más datos, entrenamiento en GPU)

Curso de Deep Learning · Especialización en Inteligencia Artificial · Universidad Nacional de Colombia

**Autores:** Thomas Monnier Granda, Esteban Felipe Genes Noya, Juan Sebastian Higuita Riascos, Alexandra Arias Mazo y Eliel Segura Perez

> **Pregunta de este cuaderno:** ¿qué modelos quedaron elegidos, dónde están guardados y cómo construyen la respuesta conjunta?

Este cuaderno no entrena nada: lee las evidencias que dejaron los pasos 1 a 5 (`iteracion_2/*.bat`) y las explica. Cada cifra sale de un archivo del repositorio.

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import apoyo  # iteracion_2/notebooks/apoyo.py: rutas, lectura de evidencias y figuras

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
apoyo.estilo()
print("Raíz del repositorio:", apoyo.RAIZ.name, "| evidencias en", apoyo.EVIDENCIAS.relative_to(apoyo.RAIZ))

Raíz del repositorio: Proyecto_Final | evidencias en iteracion_2/evidencias


## 1. Los modelos elegidos

Para cada rama y cada partición se eligió la red y la configuración con menor NLL en la validación de
esa partición. La rama 2 eligió además entre sus tres agregadores. Los codificadores elegidos y sus
vectores están en `iteracion_2/modelos_finales/<rama>/v1/`, cada uno con su huella SHA-256.

In [2]:
filas = []
for rama in ("vistas", "transitos", "escalares"):
    manifiesto = apoyo.leer_json(apoyo.MODELOS / rama / "v1" / "manifiesto.json")
    for k, e in manifiesto["particiones"].items():
        filas.append({"rama": rama, "partición": int(k), "red": apoyo.REDES[e["red"]],
                      "configuración": e["configuracion"], "parámetros": e["parametros"],
                      "NLL de validación": round(e["puntaje_validacion"], 4)})
elegidos = pd.DataFrame(filas)
display(elegidos.pivot(index="partición", columns="rama", values="red"))
display(elegidos.set_index(["rama", "partición"]))

rama,escalares,transitos,vistas
partición,,,
0,Rama 4 · escalares,Rama 2 · atención,Rama 1 · vistas plegadas
1,Rama 4 · escalares,Rama 2 · promedio,Rama 1 · vistas plegadas
2,Rama 4 · escalares,Rama 2 · promedio,Rama 1 · vistas plegadas
3,Rama 4 · escalares,Rama 2 · atención,Rama 1 · vistas plegadas
4,Rama 4 · escalares,Rama 2 · GRU,Rama 1 · vistas plegadas


red configuración  parámetros  \
rama      partición                                                       
vistas    0          Rama 1 · vistas plegadas         cfg00       57747   
          1          Rama 1 · vistas plegadas         cfg00       57747   
          2          Rama 1 · vistas plegadas         cfg01      224931   
          3          Rama 1 · vistas plegadas         cfg00       57747   
          4          Rama 1 · vistas plegadas         cfg00       57747   
transitos 0                 Rama 2 · atención         cfg07       22628   
          1                 Rama 2 · promedio         cfg10       57443   
          2                 Rama 2 · promedio         cfg10       57443   
          3                 Rama 2 · atención         cfg00       13604   
          4                      Rama 2 · GRU         cfg00       18723   
escalares 0                Rama 4 · escalares         cfg05        2307   
          1                Rama 4 · escalares         cfg06        1123   
          2                Rama 4 · escalares         cfg01        4547   
          3                Rama 4 · escalares         cfg05        2307   
          4                Rama 4 · escalares         cfg01        4547   

                     NLL de validación  
rama      partición                     
vistas    0                     0.1659  
          1                     0.2061  
          2                     0.1819  
          3                     0.1807  
          4                     0.1935  
transitos 0                     0.4005  
          1                     0.4328  
          2                     0.4456  
          3                     0.4014  
          4                     0.4301  
escalares 0                     0.1216  
          1                     0.1357  
          2                     0.1160  
          3                     0.1031  
          4                     0.1214

**Lectura.** En la rama 2 ningún agregador domina: cada partición prefiere uno distinto. Coincide con
lo que mostró la búsqueda: promedio, atención y GRU rinden parecido con estos datos.

## 2. Qué hay guardado y cómo volver a usarlo

Cada carpeta tiene, por partición: el codificador (`particion_k.keras`), los vectores de todos los
ejemplos (`vectores_particion_k.npz`) y, si aplica, la estandarización usada. Las cabezas de fusión están
en `evidencias/resultados/fusion/<configuración>/cabezas/`.

In [3]:
for rama in ("vistas", "transitos", "escalares"):
    archivos = sorted(p.name for p in (apoyo.MODELOS / rama / "v1").iterdir())
    print(f"{rama:10s}", len(archivos), "archivos ·", ", ".join(archivos[:4]), "…")
with np.load(apoyo.MODELOS / "vistas" / "v1" / "vectores_particion_0.npz") as d:
    print("\nVectores de la rama 1, partición 0:", d["z"].shape, "(ejemplos, dimensiones)")

vistas     11 archivos · manifiesto.json, particion_0.keras, particion_1.keras, particion_2.keras …
transitos  16 archivos · estandarizacion_particion_0.npz, estandarizacion_particion_1.npz, estandarizacion_particion_2.npz, estandarizacion_particion_3.npz …
escalares  16 archivos · estandarizacion_particion_0.npz, estandarizacion_particion_1.npz, estandarizacion_particion_2.npz, estandarizacion_particion_3.npz …

Vectores de la rama 1, partición 0: (8942, 32) (ejemplos, dimensiones)


Para cargar un codificador (requiere Keras ≥ 3.15, la versión con la que se guardaron):

```python
import keras
from temporal import modelos          # registra las capas propias
codificador = keras.saving.load_model("iteracion_2/modelos_finales/transitos/v1/particion_0.keras",
                                      compile=False, safe_mode=True)
```

## 3. La fusión, sin la rama de imagen

La cabeza recibe los vectores de las ramas 1, 2 y 4 y una bandera por rama. Se entrenó con cinco
semillas. Además de la fusión completa se entrenaron variantes para saber **qué aporta cada parte**:

In [4]:
filas = []
for nombre in ("tres_ramas", "tres_ramas_ajuste_cruzado", "solo_curvas", "solo_vistas", "solo_transitos", "solo_escalares"):
    m = apoyo.leer_json(apoyo.RESULTADOS / "fusion" / nombre / "metricas.json")
    g = m["semilla_0"]["global"]
    filas.append({"configuración": apoyo.FUSIONES[nombre], "ramas": " + ".join(m["ramas"]),
                  "AUC por semilla": " ".join(f"{a:.4f}" for a in m["auc_por_semilla"]),
                  "desviación entre semillas": m["auc_desviacion_semillas"],
                  "aciertos (semilla 0)": g["exactitud"]["estimacion"], "NLL": g["nll"]["estimacion"]})
display(pd.DataFrame(filas).set_index("configuración"))

,ramas,AUC por semilla,desviación entre semillas,aciertos (semilla 0),NLL
configuración,,,,,
Fusión de tres ramas,vistas + transitos + escalares,0.9934 0.9941 0.9935 0.9940 0.9934,0.0003,0.9695,0.0735
Fusión con ajuste cruzado,vistas + transitos + escalares,0.9930 0.9927 0.9929 0.9930 0.9930,0.0001,0.9696,0.0796
Solo curvas (ramas 1 y 2),vistas + transitos,0.9694 0.9702 0.9700 0.9702 0.9702,0.0003,0.9364,0.1618
Solo rama 1,vistas,0.9544 0.9538 0.9540 0.9540 0.9543,0.0002,0.9274,0.1872
Solo rama 2,transitos,0.9463 0.9468 0.9461 0.9465 0.9459,0.0003,0.8265,0.4438
Solo rama 4,escalares,0.9828 0.9825 0.9827 0.9830 0.9823,0.0002,0.9541,0.1189


La variación entre semillas de la cabeza es mínima: el resultado no depende de la suerte del arranque.

## 4. Un riesgo conocido: el apilado

La cabeza se entrena con vectores de ejemplos que los codificadores **ya vieron**, y se evalúa con
vectores de ejemplos que no vieron. Si los codificadores memorizaron, la cabeza aprende de vectores
demasiado buenos. Se mide comparando su pérdida en entrenamiento y en validación, y se prueba un
remedio (*ajuste cruzado*: la cabeza solo ve vectores que los codificadores no usaron).

In [5]:
m = apoyo.leer_json(apoyo.RESULTADOS / "fusion" / "tres_ramas" / "metricas.json")
print("Apilado en la fusión de tres ramas:", m["apilado"])
dif = apoyo.comparacion()["diferencias"]
clave = next(k for k in dif if "efecto del apilado" in k)
print("Fusión normal − fusión con ajuste cruzado (AUC):", apoyo.con_intervalo(dif[clave], signo=True),
      "→ distinguible:", apoyo.distinguible(dif[clave]))

Apilado en la fusión de tres ramas: {'nll_entrenamiento': 0.0382, 'nll_validacion': 0.0756, 'razon': 1.98, 'alerta': False}
Fusión normal − fusión con ajuste cruzado (AUC): +0.0005 [-0.0004; +0.0015] → distinguible: no


**Lectura.** La pérdida de validación es cerca del doble de la de entrenamiento: los codificadores sí
memorizan algo. Pero el remedio no cambia el AUC de forma distinguible, así que el resultado principal
no está inflado por el apilado.

## 5. Y la rama de imagen

No se inventó ni se rellenó: su bandera vale 0 y su vector no entra. Para sumarla basta con poner sus
vectores en `modelos_finales/imagen/<versión>/` con el mismo formato y volver a entrenar la cabeza
(minutos). Ningún codificador de este curso cambia.